# RamanSPy × 食品分析入門：用模擬食品拉曼光譜學 PCA 與分類

對象：沒有程式基礎的食品營養系大學生。

學習目標：
1. 看懂 `sample × Raman shift` 的資料表。
2. 知道前處理（平滑、基線校正、標準化）為什麼重要。
3. 用 PCA score plot 判讀食品樣品是否分群。
4. 用簡單分類模型評估是否能辨識食品類別。

> 小提醒：第一次執行請從上到下逐格按 ▶。

In [ ]:
# 如果在 Google Colab，取消下一行註解可安裝 RamanSPy
# !pip -q install ramanspy

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.signal import savgol_filter
from sklearn.decomposition import PCA
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, ConfusionMatrixDisplay

np.random.seed(42)
plt.rcParams['figure.dpi'] = 120
print('環境準備完成')

## 1. 產生三種食品樣品的模擬 Raman 光譜

我們先不用真實資料，因為初學者最容易卡在下載與格式。這裡用幾個常見食品成分的「概念峰」建立三類樣品：

- **starch-rich**：澱粉/碳水化合物訊號較強
- **protein-rich**：蛋白質相關訊號較強
- **fat-adulterated**：脂肪/油脂相關訊號較強

這不是標準品資料，只用來練習資料分析流程。

In [ ]:
axis = np.linspace(500, 1800, 650)

def gaussian(x, center, width, height):
    return height * np.exp(-0.5 * ((x-center)/width)**2)

base_peaks = {
    'starch': [(480, 20, .45), (940, 22, .70), (1125, 35, .55), (1340, 28, .35)],
    'protein': [(760, 25, .35), (1004, 18, .45), (1450, 30, .50), (1660, 36, .62)],
    'fat': [(1065, 24, .35), (1300, 28, .45), (1445, 26, .62), (1745, 34, .70)],
}

def make_spectrum(kind):
    y = np.zeros_like(axis)
    weights = {
        'starch-rich': {'starch':1.25, 'protein':0.45, 'fat':0.25},
        'protein-rich': {'starch':0.55, 'protein':1.25, 'fat':0.35},
        'fat-adulterated': {'starch':0.55, 'protein':0.35, 'fat':1.30},
    }[kind]
    for comp, peaks in base_peaks.items():
        for center, width, height in peaks:
            jitter = np.random.normal(0, 3)
            scale = weights[comp] * np.random.normal(1, 0.08)
            y += gaussian(axis, center+jitter, width, height*scale)
    baseline = 0.15 + 0.00015*(axis-500) + 0.12*np.sin(axis/260 + np.random.rand())
    noise = np.random.normal(0, 0.025, size=axis.size)
    return y + baseline + noise

labels = []
X = []
for kind in ['starch-rich', 'protein-rich', 'fat-adulterated']:
    for _ in range(36):
        X.append(make_spectrum(kind))
        labels.append(kind)
X = np.array(X)
labels = np.array(labels)

df = pd.DataFrame(X, columns=[f'{v:.1f}' for v in axis])
df.insert(0, 'label', labels)
df.head()

## 2. 畫出原始光譜

觀察：有沒有基線傾斜？類別平均是否有不同？雜訊是否會影響判讀？

In [ ]:
colors = {'starch-rich':'#1769ff', 'protein-rich':'#0f9f6e', 'fat-adulterated':'#f59e0b'}
plt.figure(figsize=(9,5))
for kind in colors:
    subset = X[labels==kind]
    for row in subset[:8]:
        plt.plot(axis, row, color=colors[kind], alpha=.12)
    plt.plot(axis, subset.mean(axis=0), color=colors[kind], lw=2.5, label=kind)
plt.xlabel('Raman shift (cm$^{-1}$)')
plt.ylabel('Intensity (a.u.)')
plt.title('Raw simulated food Raman spectra')
plt.legend(); plt.show()

## 3. 簡易前處理：平滑、扣基線、標準化

1. Savitzky–Golay 平滑：降低雜訊。
2. 用低百分位數估計基線並扣除：降低背景漂移。
3. Min-max normalization：讓不同樣品強度落在 0–1，方便比較形狀。

In [ ]:
def preprocess_matrix(X):
    smooth = savgol_filter(X, window_length=17, polyorder=3, axis=1)
    baseline = np.percentile(smooth, 8, axis=1, keepdims=True)
    corrected = smooth - baseline
    corrected = np.clip(corrected, 0, None)
    minv = corrected.min(axis=1, keepdims=True)
    maxv = corrected.max(axis=1, keepdims=True)
    return (corrected - minv) / (maxv - minv + 1e-12)

Xp = preprocess_matrix(X)

plt.figure(figsize=(9,5))
for kind in colors:
    subset = Xp[labels==kind]
    plt.plot(axis, subset.mean(axis=0), color=colors[kind], lw=2.5, label=kind)
plt.xlabel('Raman shift (cm$^{-1}$)')
plt.ylabel('Normalized intensity')
plt.title('Mean spectra after simple preprocessing')
plt.legend(); plt.show()

## 4. 如果 RamanSPy 可用：把資料放進 RamanSPy 的視覺化流程

RamanSPy 官方文件說明可用 `rp.plot.spectra(spectra, wavenumber_axis=axis)` 畫光譜。這格會自動偵測是否已安裝 RamanSPy；若未安裝，不影響後續 PCA。

In [ ]:
try:
    import ramanspy as rp
    print('RamanSPy version:', getattr(rp, '__version__', 'unknown'))
    rp.plot.spectra([Xp[labels=='starch-rich'][:5], Xp[labels=='protein-rich'][:5], Xp[labels=='fat-adulterated'][:5]],
                    wavenumber_axis=axis, plot_type='single', label=['starch', 'protein', 'fat'])
    plt.show()
except Exception as e:
    print('這個環境目前沒有成功使用 RamanSPy；先用 NumPy/scikit-learn 完成核心流程。')
    print('原因：', repr(e))

## 5. PCA：把 650 個波數變成 2 個座標

PCA 不是魔法，它只是找出最能解釋光譜差異的方向。每個點是一個食品樣品；點靠近代表整條光譜 pattern 相似。

In [ ]:
pca = PCA(n_components=2)
scores = pca.fit_transform(StandardScaler().fit_transform(Xp))
plt.figure(figsize=(7,6))
for kind in colors:
    m = labels==kind
    plt.scatter(scores[m,0], scores[m,1], label=kind, s=42, alpha=.85, color=colors[kind])
plt.axhline(0, color='#94a3b8', lw=.8); plt.axvline(0, color='#94a3b8', lw=.8)
plt.xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.1f}%)')
plt.ylabel(f'PC2 ({pca.explained_variance_ratio_[1]*100:.1f}%)')
plt.title('PCA score plot')
plt.legend(); plt.show()

## 6. 小分類任務：模型能不能辨識三種食品？

這裡用線性 SVM 做最簡單的監督式分類。重點不是追求最高分，而是理解訓練集與測試集的差異。

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(Xp, labels, test_size=0.30, random_state=7, stratify=labels)
clf = make_pipeline(StandardScaler(), SVC(kernel='linear'))
clf.fit(X_train, y_train)
pred = clf.predict(X_test)
acc = accuracy_score(y_test, pred)
print(f'測試集 accuracy = {acc:.3f}')
ConfusionMatrixDisplay.from_predictions(y_test, pred, xticks_rotation=25, cmap='Blues')
plt.title('Confusion matrix'); plt.show()

## 7. 學習檢核問題

1. 為什麼前處理後的光譜比較適合做 PCA？
2. PCA 圖上三群分開，代表食品樣品有哪些可能差異？
3. 如果未來換成真實食品資料，哪些步驟最可能需要調整？
4. 為什麼不能只用一個波峰就宣稱某個成分一定比較高？

完成後，回到 HTML 頁面的互動測驗取得即時分數。